# SmartVision AI — Phase 4: Model comparison, inference pipeline, quantization

**Colab without switching runtimes:** use [`retrain_all_colab.ipynb`](retrain_all_colab.ipynb). This notebook is still the standalone comparison pass.

Merges classification + YOLO metrics, builds comparison charts, runs the end-to-end pipeline (YOLO → optional CNN verify → NMS/conf filter), and exports a lighter graph for deployment.

**If you run this file alone:** do it after notebooks 02 and 03 finish so it reads the new `classification_metrics.json` and `yolo_metrics.json` already on Drive (`MyDrive/SmartVision_artifacts/reports/`). Do not rebuild the dataset.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip -q install tensorflow ultralytics pandas matplotlib seaborn pillow opencv-python-headless pyyaml

In [ ]:
import os, sys, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/Smart_Vision_AI")
    OUT_ROOT = Path("/content/drive/MyDrive/SmartVision_artifacts")
    sys.path.insert(0, str(Path("/content/drive/MyDrive/Smart_Vision_AI")))
    sys.path.insert(0, str(PROJECT_ROOT))
else:
    PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
    OUT_ROOT = PROJECT_ROOT
    sys.path.insert(0, str(PROJECT_ROOT))

REPORTS = OUT_ROOT / "reports"
FIGURES = REPORTS / "figures"
MODELS = OUT_ROOT / "models"
FIGURES.mkdir(parents=True, exist_ok=True)

clf = json.loads((REPORTS / "classification_metrics.json").read_text(encoding="utf-8"))
yolo = json.loads((REPORTS / "yolo_metrics.json").read_text(encoding="utf-8"))
meta_path = PROJECT_ROOT / "smartvision_dataset" / "dataset_metadata.json"
meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.exists() else {}
print("best CNN:", clf["best_classification_model"])
print("YOLO mAP50 val:", yolo["val"]["map50"])

In [ ]:
## Comparison table + charts

rows = []
for name, m in clf["classification"].items():
    rows.append({
        "model": name,
        "task": "classification",
        "accuracy": m["accuracy"],
        "precision": m["precision_macro"],
        "recall": m["recall_macro"],
        "f1": m["f1_macro"],
        "top5": m["top5_accuracy"],
        "inference_ms": m["inference_ms"],
        "size_mb": m["model_size_mb"],
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
df.plot.bar(x="model", y=["accuracy", "f1", "top5"], ax=axes[0], rot=20)
axes[0].set_ylim(0, 1); axes[0].set_title("Accuracy / F1 / Top-5")
df.plot.bar(x="model", y="inference_ms", ax=axes[1], rot=20, legend=False, color="#2E86AB")
axes[1].set_title("Inference ms / image")
df.plot.bar(x="model", y="size_mb", ax=axes[2], rot=20, legend=False, color="#E94F37")
axes[2].set_title("Model size (MB)")
fig.tight_layout()
fig.savefig(FIGURES / "model_comparison.png", dpi=140)
plt.show()

# Speed vs accuracy scatter
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(df["inference_ms"], df["accuracy"], s=df["size_mb"] * 8)
for _, r in df.iterrows():
    ax.annotate(r["model"], (r["inference_ms"], r["accuracy"]), textcoords="offset points", xytext=(6, 4))
ax.set_xlabel("inference ms"); ax.set_ylabel("test accuracy")
ax.set_title("Accuracy–speed tradeoff (bubble ~ size)")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / "accuracy_speed_tradeoff.png", dpi=140)
plt.show()

best = clf["best_classification_model"]
print("Selected classification model (highest test accuracy):", best)
print("YOLO is used for multi-object localization; CNN is optional verification on crops.")

In [ ]:
## End-to-end pipeline on detection test images
# YOLO NMS is internal; we filter confidence > 0.50 as the brief requires.

from ultralytics import YOLO

CLASS_NAMES = clf["class_names"]
yolo_model = YOLO(str(MODELS / "yolov8_best.pt"))
cnn_name = best
cnn_path = {
    "VGG16": MODELS / "vgg16.keras",
    "ResNet50": MODELS / "resnet50.keras",
    "MobileNetV2": MODELS / "mobilenetv2.keras",
    "EfficientNetB0": MODELS / "efficientnetb0.keras",
}[cnn_name]

import tensorflow as tf
cnn = tf.keras.models.load_model(cnn_path, compile=False)

DET = PROJECT_ROOT / "smartvision_dataset" / "detection"
if not (DET / "images" / "test").exists():
    DET = Path("/content/smartvision_dataset/detection")
test_imgs = sorted((DET / "images" / "test").glob("*.jpg"))[:6]

def ensure_rgb(im):
    return im.convert("RGB") if im.mode != "RGB" else im

def classify_crop(pil_crop):
    arr = np.asarray(pil_crop.resize((224, 224)), dtype=np.float32)
    pred = cnn.predict(np.expand_dims(arr, 0), verbose=0)[0]
    i = int(pred.argmax())
    return CLASS_NAMES[i], float(pred[i])

def run_pipeline(path, conf=0.50, verify_cnn=True):
    im = ensure_rgb(Image.open(path))
    res = yolo_model.predict(source=np.asarray(im), conf=conf, iou=0.45, verbose=False)[0]
    dets = []
    if res.boxes is not None:
        for b in res.boxes:
            x1, y1, x2, y2 = [float(v) for v in b.xyxy[0].tolist()]
            cls_id = int(b.cls[0]); score = float(b.conf[0])
            label = res.names.get(cls_id, CLASS_NAMES[cls_id])
            rec = {"xyxy": [x1, y1, x2, y2], "class": label, "confidence": score, "cnn_class": None, "agree": None}
            if verify_cnn:
                crop = im.crop((max(0,int(x1)), max(0,int(y1)), int(x2), int(y2)))
                if crop.size[0] > 2 and crop.size[1] > 2:
                    cc, cs = classify_crop(crop)
                    rec["cnn_class"] = cc
                    rec["cnn_confidence"] = cs
                    rec["agree"] = cc == label
            dets.append(rec)
    canvas = im.copy()
    dr = ImageDraw.Draw(canvas)
    for d in dets:
        x1, y1, x2, y2 = d["xyxy"]
        dr.rectangle([x1, y1, x2, y2], outline="red", width=3)
        tag = f"{d['class']} {d['confidence']:.2f}"
        if d.get("cnn_class"):
            tag += f" | CNN {d['cnn_class']} {d['cnn_confidence']:.2f}"
        dr.text((x1, max(0, y1 - 12)), tag, fill="red")
    return canvas, dets

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
all_dets = []
t0 = time.perf_counter()
for ax, p in zip(axes.ravel(), test_imgs):
    canvas, dets = run_pipeline(p, conf=0.50, verify_cnn=True)
    all_dets.extend(dets)
    ax.imshow(canvas)
    ax.set_title(f"{p.name}  n={len(dets)}", fontsize=8)
    ax.axis("off")
elapsed = time.perf_counter() - t0
fig.suptitle(f"Pipeline YOLO+{cnn_name}  conf>0.5  ({elapsed/max(len(test_imgs),1):.2f}s / image)")
fig.tight_layout()
fig.savefig(FIGURES / "pipeline_samples.png", dpi=140)
plt.show()
if all_dets:
    agree = [d["agree"] for d in all_dets if d["agree"] is not None]
    print(f"CNN-YOLO agreement on {len(agree)} crops: {np.mean(agree):.3f}" if agree else "no crops")
print("objects/image (this slice):", len(all_dets) / max(len(test_imgs), 1))

In [ ]:
## Quantization / export for cloud (Phase 4.3)
# TFLite dynamic-range for the best CNN; ONNX for YOLO if export succeeds.

best_path = {
    "VGG16": MODELS / "vgg16.keras",
    "ResNet50": MODELS / "resnet50.keras",
    "MobileNetV2": MODELS / "mobilenetv2.keras",
    "EfficientNetB0": MODELS / "efficientnetb0.keras",
}[clf["best_classification_model"]]

converter = tf.lite.TFLiteConverter.from_keras_model(tf.keras.models.load_model(best_path, compile=False))
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_bytes = converter.convert()
tflite_path = MODELS / "best_cnn_dynamic.tflite"
tflite_path.write_bytes(tflite_bytes)
print("TFLite", tflite_path, "MB", len(tflite_bytes)/1e6)

try:
    YOLO = __import__("ultralytics").YOLO
    y = YOLO(str(MODELS / "yolov8_best.pt"))
    y.export(format="onnx", imgsz=640, simplify=True)
    print("ONNX export attempted in the YOLO run directory / models folder")
except Exception as e:
    print("ONNX export skipped:", e)

quant = {
    "tflite": str(tflite_path),
    "tflite_mb": len(tflite_bytes) / 1e6,
    "original_mb": clf["classification"][clf["best_classification_model"]]["model_size_mb"],
}
print(quant)

In [ ]:
## Final metrics.json consumed by the Streamlit Performance page

metrics = {
    "class_names": clf["class_names"],
    "dataset": meta.get("classification", {}),
    "dataset_decisions": meta.get("decisions", {}),
    "classification": clf["classification"],
    "best_classification_model": clf["best_classification_model"],
    "yolo": {
        "model": yolo.get("model", "YOLOv8s"),
        "map50": yolo["val"]["map50"],
        "map50_95": yolo["val"]["map50_95"],
        "precision": yolo["val"]["precision"],
        "recall": yolo["val"]["recall"],
        "fps": yolo.get("fps"),
        "test": yolo.get("test", {}),
        "per_class_ap50": yolo.get("per_class_ap50", {}),
        "meets_map50_floor": yolo.get("meets_map50_floor"),
    },
    "quantization": quant,
}
(REPORTS / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("Wrote", REPORTS / "metrics.json")
print("Download Drive/SmartVision_artifacts/{models,reports} into the GitHub repo.")